In [ ]:
# [목적] 임베딩 기반 의미 유사도 예제 선택기를 이용한 Few-shot Prompt 흐름을 실습합니다.
# 질문의 의미를 숫자 벡터로 비교해 가장 비슷한 예시만 고르고, 그 예시를 모델의 답변 참고 자료로 사용합니다.
# 예시 저장소와 선택기를 여기서 준비하며, 결과는 뒤 셀의 프롬프트 생성과 모델 호출에 이어집니다.

from langchain_core.example_selectors import (
    MaxMarginalRelevanceExampleSelector,
    SemanticSimilarityExampleSelector,
)

from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# 질문과 답변 예시는 의미가 비슷한 예시를 찾을 때 비교 대상으로 사용합니다.
examples = [
    {
        "question": "스티브 잡스와 빌 게이츠 중 누가 더 오래 살았나요?",
        "answer": "스티브 잡스는 56세에 사망했고, 빌 게이츠는 그보다 오래 살았습니다.",
    },
    {
        "question": "에펠탑과 롯데월드타워 중 어느 것이 더 높은가요?",
        "answer": "롯데월드타워가 에펠탑보다 더 높습니다.",
    },
]

# Chroma는 예시의 임베딩을 저장하고 유사한 내용을 빠르게 검색하는 벡터 저장소입니다.
chroma = Chroma("examples_selector", OpenAIEmbeddings())

# k=1은 새 질문과 의미가 가장 가까운 예시 한 개만 선택한다는 뜻입니다.
example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples,
    OpenAIEmbeddings(),
    Chroma,
    k=1,
)

In [ ]:
# [목적] 새 질문과 가장 의미가 비슷한 예시가 제대로 선택되는지 확인합니다.
# 앞 셀의 example_selector에 질문을 전달하고, 선택된 질문·답변을 출력해 검색 결과를 눈으로 검증합니다.

question = "Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?"
# select_examples는 질문을 임베딩으로 바꾼 뒤 저장된 예시들과 의미적 거리를 비교합니다.
selected_examples = example_selector.select_examples({"question": question})
print(f"입력에 가장 유사한 예시:\n{question}\n")
# 선택 결과는 목록이므로 반복문으로 각 예시의 질문과 답변을 꺼냅니다.
for example in selected_examples:
    print(f'question:\n{example["question"]}')
    print(f'answer:\n{example["answer"]}')

In [ ]:
# [목적] 선택된 예시와 새 질문을 합쳐 모델에 전달할 Few-shot Prompt를 완성합니다.
# example_prompt는 예시 형식을 정하고, FewShotPromptTemplate은 선택기가 고른 예시를 새 질문 앞에 자동으로 붙입니다.
# 완성된 example_selector_prompt를 출력해 실제 모델 입력 모양을 미리 확인합니다.

from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

# 각 예시 딕셔너리의 question과 answer가 들어갈 자리표시자 형식입니다.
example_prompt = PromptTemplate.from_template(
    "Question:\n{question}\nAnswer:\n{answer}"
)

# 고정 예시 대신 example_selector를 연결해 질문마다 알맞은 예시가 달라지게 합니다.
prompt = FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt,
    suffix="Question: \n{question}\nAnswer:",
    input_variables=["question"],
)

question = "Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?"
# format이 예시 선택과 문자열 조립을 실행해 최종 프롬프트를 만듭니다.
example_selector_prompt = prompt.format(question=question)
print(example_selector_prompt)

In [ ]:
# [목적] 의미가 비슷한 예시를 자동으로 고르는 프롬프트를 ChatOpenAI와 연결해 답변을 생성합니다.
# 질문이 들어오면 예시 선택 → 프롬프트 완성 → 모델 호출 순서로 실행되고, 응답은 화면에 실시간 출력됩니다.

from langchain_openai import ChatOpenAI
from langchain_teddynote.messages import stream_response

# ChatOpenAI 객체가 완성된 프롬프트를 받아 실제 답변을 생성합니다.
llm = ChatOpenAI()

# 앞에서 만든 선택기를 다시 연결해 질문마다 관련 예시를 동적으로 포함합니다.
prompt = FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt,
    suffix="Question: \n{question}\nAnswer:",
    input_variables=["question"],
)

# | 연산자는 프롬프트의 결과를 모델 입력으로 넘기는 실행 흐름을 만듭니다.
chain = prompt | llm  # 체인 생성

# stream은 완성된 답변을 기다리지 않고 생성되는 조각부터 순서대로 돌려줍니다.
answer = chain.stream(  # 결과 출력
    {"question": "Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?"}
)

stream_response(answer)